In [2]:
import numpy as np
import urllib.request
from embedder import Embedder

# Load the sentence embedding model
embedder = Embedder("models/Xenova/all-MiniLM-L6-v2")

# Example documents and a query
documents = [
    "Approximate nearest neighbor search finds similar vectors quickly.",
    "Vector databases store embeddings for fast semantic retrieval.",
    "Transformers convert text into dense numerical vectors.",
    "Retrieval augmented generation improves answers with external context.",
]

# Embed the documents and query
doc_vectors = embedder.encode_batch(documents)
query = "How does approximate nearest neighbor search work?"
query_vector = embedder.encode(query)

# Compute similarity scores and rank them
scores = doc_vectors @ query_vector
ranked_indices = np.argsort(scores)[::-1]

print("Top matches:")
for idx in ranked_indices:
    print(f"{idx}: {scores[idx]:.3f} - {documents[idx]}")

# Compare the query to the lesson content from the course repository
lesson_url = "https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/02-vector-search/lessons/07-sqlitesearch-vector.md"
lesson_text = urllib.request.urlopen(lesson_url).read().decode("utf-8")
lesson_vector = embedder.encode(lesson_text)
cosine_similarity = float(query_vector @ lesson_vector)

print(f"Cosine similarity with lesson content: {cosine_similarity:.3f}")


Top matches:
0: 0.718 - Approximate nearest neighbor search finds similar vectors quickly.
1: 0.293 - Vector databases store embeddings for fast semantic retrieval.
2: 0.142 - Transformers convert text into dense numerical vectors.
3: 0.113 - Retrieval augmented generation improves answers with external context.
Cosine similarity with lesson content: 0.361


In [ ]:
# Optional: a simple sanity check
print("Notebook is ready for vector search experiments.")


In [ ]:
# Optional: load lesson markdown files from the GitHub repository
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

lesson_documents = [file.parse() for file in reader.read()]
print(f"Loaded {len(lesson_documents)} lesson documents.")


In [6]:
from gitsource import chunk_documents

# Convert the plain strings into the dictionary shape expected by the chunker
chunkable_documents = [{"content": text} for text in documents]

# Chunk the current document list into overlapping windows
chunks = chunk_documents(chunkable_documents, size=2000, step=1000)
print(f"Created {len(chunks)} chunks.")
for i, chunk in enumerate(chunks[:5]):
    print(f"Chunk {i+1}: {chunk['content'][:120]}...")


Created 4 chunks.
Chunk 1: Approximate nearest neighbor search finds similar vectors quickly....
Chunk 2: Vector databases store embeddings for fast semantic retrieval....
Chunk 3: Transformers convert text into dense numerical vectors....
Chunk 4: Retrieval augmented generation improves answers with external context....


In [10]:
from gitsource import GithubRepositoryDataReader, chunk_documents
import numpy as np
from embedder import Embedder

embedder = Embedder("models/Xenova/all-MiniLM-L6-v2")
reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path and "02-vector-search/lessons/" in path,
)

files = list(reader.read())
docs = [{"filename": f.filename, "content": f.content} for f in files]
chunks = chunk_documents(docs, size=2000, step=1000)

texts = [c["content"] for c in chunks]
X = embedder.encode_batch(texts)
query = "How does approximate nearest neighbor search work?"
v = embedder.encode(query)
scores = X @ v
best_idx = int(np.argmax(scores))

print(chunks[best_idx]["filename"])
print(float(scores[best_idx]))


02-vector-search/lessons/07-sqlitesearch-vector.md
0.6489016436447387


In [13]:
from minsearch import VectorSearch
import numpy as np

# Build a vector search index over the lesson chunks
vector_index = VectorSearch()

# Prepare vectors and payload documents
vectors = np.array([embedder.encode(chunk["content"]) for chunk in chunks])
payload = [
    {
        "filename": chunk["filename"],
        "content": chunk["content"],
    }
    for chunk in chunks
]

vector_index.fit(vectors, payload)

query = "What metric do we use to evaluate a search engine?"
query_vector = embedder.encode(query)
results = vector_index.search(query_vector, num_results=5)

print("Vector search results:")
for item in results:
    print(item["filename"])


Vector search results:
02-vector-search/lessons/01-intro.md
02-vector-search/lessons/07-sqlitesearch-vector.md
02-vector-search/lessons/02-embeddings.md
02-vector-search/lessons/10-next-steps.md
02-vector-search/lessons/01-intro.md


In [17]:
from minsearch import Index, VectorSearch
import numpy as np
from gitsource import GithubRepositoryDataReader, chunk_documents
from embedder import Embedder

embedder = Embedder("models/Xenova/all-MiniLM-L6-v2")

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = list(reader.read())
docs = [{"filename": f.filename, "content": f.content} for f in files]
chunks = chunk_documents(docs, size=2000, step=1000)

payload = [{"filename": chunk["filename"], "content": chunk["content"]} for chunk in chunks]
vectors = np.array([embedder.encode(chunk["content"]) for chunk in chunks])

text_index = Index(text_fields=["content"])
text_index.fit(payload)

vector_index = VectorSearch()
vector_index.fit(vectors, payload)

query = "How do I store vectors in PostgreSQL?"
text_results = text_index.search(query, num_results=5)
vector_results = vector_index.search(embedder.encode(query), num_results=5)

print("Text search results:")
for item in text_results:
    print(item["filename"])

print("\nVector search results:")
for item in vector_results:
    print(item["filename"])

text_filenames = {item["filename"] for item in text_results}
vector_only = [
    item["filename"]
    for item in vector_results
    if item["filename"] not in text_filenames
]
print("\nFiles only in vector results:")
for filename in vector_only:
    print(filename)


Text search results:
02-vector-search/lessons/02-embeddings.md
03-orchestration/lessons/05-rag.md
02-vector-search/lessons/01-intro.md
03-orchestration/lessons/05-rag.md
02-vector-search/lessons/01-intro.md

Vector search results:
02-vector-search/lessons/08-pgvector.md
02-vector-search/lessons/08-pgvector.md
03-orchestration/lessons/05-rag.md
02-vector-search/lessons/08-pgvector.md
02-vector-search/lessons/08-pgvector.md

Files only in vector results:
02-vector-search/lessons/08-pgvector.md
02-vector-search/lessons/08-pgvector.md
02-vector-search/lessons/08-pgvector.md
02-vector-search/lessons/08-pgvector.md


In [22]:
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = doc.get("filename") or doc.get("content") or str(rank)
            if isinstance(key, tuple):
                key = key[0]
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]


In [23]:
query = "How do I give the model access to tools?"

text_results = text_index.search(query, num_results=5)
vector_results = vector_index.search(embedder.encode(query), num_results=5)

fused_results = rrf([vector_results, text_results], k=60, num_results=5)

print("Text search results:")
for item in text_results:
    print(item["filename"])

print("\nVector search results:")
for item in vector_results:
    print(item["filename"])

print("\nRRF fused results:")
for item in fused_results:
    print(item["filename"])


Text search results:
01-agentic-rag/lessons/14-agentic-loop.md
01-agentic-rag/lessons/13-function-calling.md
01-agentic-rag/lessons/13-function-calling.md
01-agentic-rag/lessons/13-function-calling.md
04-evaluation/lessons/02-ground-truth.md

Vector search results:
01-agentic-rag/lessons/01-intro.md
04-evaluation/lessons/02-ground-truth.md
01-agentic-rag/lessons/16-other-frameworks.md
01-agentic-rag/lessons/15-frameworks.md
01-agentic-rag/lessons/13-function-calling.md

RRF fused results:
01-agentic-rag/lessons/13-function-calling.md
04-evaluation/lessons/02-ground-truth.md
01-agentic-rag/lessons/01-intro.md
01-agentic-rag/lessons/14-agentic-loop.md
01-agentic-rag/lessons/16-other-frameworks.md
